# Cifar cnn pytorch tensorflow

Original experiment from [Neural-Networks-and-Deep-Learning-Using-Pytorch-and-Tensor-Flow](https://github.com/Muhammad-Huzifa/Neural-Networks-and-Deep-Learning-Using-Pytorch-and-Tensor-Flow/blob/dfa17b2b298631b10e5904b2f1c39ad64d424bc1/Deep%20Neural%20Networks/CNN/CIFAR%20Using%20CNN.ipynb). Run from a fresh kernel. See [dataset and environment instructions](../../docs/DATASETS.md).

Framework training, downloaded datasets, and custom image inputs require the dependencies and resources described in the collection README. This migration has checked syntax and paths; it has not repeated every training experiment.

In [ ]:
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "requirements/base.txt").is_file())
TABULAR = ROOT / "data/tabular"
IMAGES = ROOT / "data/images"
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)


In [ ]:
import torch.nn.functional as F


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

# Define transformations for data augmentation and normalization
transform = transforms.Compose([
    transforms.ToTensor(),           # Convert images to tensors
    transforms.Normalize((0.5,), (0.5,))  # Normalize to range [-1, 1]
])

# Load CIFAR-10 dataset
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=32, shuffle=True)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=32, shuffle=False)

# Define class labels
classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

# Display a sample image
sample_img, sample_label = trainset[0]
plt.imshow(sample_img.permute(1, 2, 0))  # Convert tensor to image format
plt.title(f"Class: {classes[sample_label]}")
plt.show()


In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=5, stride=1, padding=0)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=5, stride=1, padding=0)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Compute the size after convolutions (5x5x32 from earlier calculations)
        self.fc3 = nn.Linear(5*5*32, 128)
        self.fc4 = nn.Linear(128, 64)
        self.fc_out = nn.Linear(64, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # Conv1 -> ReLU -> MaxPool
        x = self.pool(F.relu(self.conv2(x)))  # Conv2 -> ReLU -> MaxPool
        x = torch.flatten(x, start_dim=1)     # Flatten
        x = F.relu(self.fc3(x))               # FC3 -> ReLU
        x = F.relu(self.fc4(x))               # FC4 -> ReLU
        x = self.fc_out(x)                    # Output layer (No activation)
        return x

# Initialize model
model = CNN()


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:
# Training loop
epochs = 5  # Number of training epochs
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(epochs):
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(trainloader):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()  # Reset gradients
        outputs = model(inputs)  # Forward pass
        loss = criterion(outputs, labels)  # Compute loss
        loss.backward()  # Backpropagation
        optimizer.step()  # Update weights

        running_loss += loss.item()
        if i % 100 == 99:  # Print loss every 100 mini-batches
            print(f"Epoch {epoch+1}, Batch {i+1}, Loss: {running_loss/100:.4f}")
            running_loss = 0.0

print("Training Finished!")


In [ ]:
correct = 0
total = 0
model.eval()  # Set model to evaluation mode (disable dropout, etc.)

with torch.no_grad():  # No need to compute gradients during testing
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)  # Get the highest probability class
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")


In [ ]:
import numpy as np

# Get a batch of test images
dataiter = iter(testloader)
images, labels = next(dataiter)

# Move images to GPU if available
images, labels = images.to(device), labels.to(device)

# Make predictions
outputs = model(images)
_, predicted = torch.max(outputs, 1)

# Display images with predictions
fig, axes = plt.subplots(1, 6, figsize=(15, 5))
for i in range(6):
    img = images[i].cpu().permute(1, 2, 0)  # Convert to numpy image format
    axes[i].imshow(img)
    axes[i].set_title(f"Predicted: {classes[predicted[i]]}")
    axes[i].axis('off')

plt.show()


In [ ]:
import tensorflow as tf

def convolutional_model(input_shape):
    """
    Implements the forward propagation for the model:
    CONV2D -> RELU -> MAXPOOL -> CONV2D -> RELU -> MAXPOOL -> FLATTEN -> DENSE

    Arguments:
    input_shape -- shape of the input dataset, e.g., (64, 64, 3)

    Returns:
    model -- TF Keras model (object containing the information for the entire training process)
    """

    input_img = tf.keras.Input(shape=input_shape)

    # CONV2D: 8 filters 4x4, stride of 1, padding 'same'
    Z1 = tf.keras.layers.Conv2D(filters=8, kernel_size=(4, 4), strides=(1, 1), padding='same')(input_img)

    # RELU
    A1 = tf.keras.layers.ReLU()(Z1)

    # MAXPOOL: window 4x4, stride 4, padding 'same'
    P1 = tf.keras.layers.MaxPooling2D(pool_size=(4, 4), strides=(4, 4), padding='same')(A1)

    # CONV2D: 16 filters 2x2, stride 1, padding 'same'
    Z2 = tf.keras.layers.Conv2D(filters=16, kernel_size=(2, 2), strides=(1, 1), padding='same')(P1)

    # RELU
    A2 = tf.keras.layers.ReLU()(Z2)

    # MAXPOOL: window 2x2, stride 2, padding 'same'
    P2 = tf.keras.layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2), padding='same')(A2)

    # FLATTEN
    F = tf.keras.layers.Flatten()(P2)

    # Dense layer with 6 neurons and softmax activation
    outputs = tf.keras.layers.Dense(units=6, activation='softmax')(F)

    # Create the model
    model = tf.keras.Model(inputs=input_img, outputs=outputs)

    return model

In [ ]:
# Define the input shape
input_shape = (64, 64, 3)  # Example input shape (height, width, channels)

# Create the model
model = convolutional_model(input_shape)

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Print the model summary
model.summary()